# 01 — Can we use this dataset for the tagging comparison?

We want to compare methods that suggest topic tags for demonstrations. Before using
any model, we need to know what records we have, what is missing, and what needs review.

**One row means one demonstration record.** A **snapshot** is a fixed copy of those
records. Its **manifest** is a small file recording the data version, row count, and
file fingerprint. An existing topic tag is a historical assignment, which may contain mistakes.

This notebook answers four questions:
1. Can we load and verify the prepared snapshot?
2. Which records were included, and what was excluded during preparation?
3. Which placeholder titles, missing places, repeated records, and rare tags need attention?
4. What must we decide before creating the model comparison's data split?

**Inputs:** `data/dataset.jsonl` and `data/manifest.json`, prepared using
[the data-preparation guide](../docs/data-preparation.md).   
**Output:** tables, a tag-frequency chart, and a private JSON quality report. Record reviewer decisions in the
[private daily audit](../docs/data-preparation.md#record-private-audit-decisions).

Use the [supported notebook environment](../docs/notebook-setup.md), then run cells from top to bottom. The code finds the repository root
from your current directory. If the snapshot is missing, follow the preparation guide.
We keep the snapshot file unchanged. EDA uses only records with at least two title words. Clear all cell outputs before committing.

**Working approach:** explore data and models in notebooks first. Shared helpers validate inputs and display tables; the analysis is visible in the cells. Service implementation follows model evaluation and selection.


## 1. Load the snapshot

The setup cell imports the shared loader and a small table-display helper. The loader
checks the file fingerprint, expected fields, unique event IDs, dates, weekdays, and
allowed tag names. Passing those checks means the file is structurally valid; it does
not establish that each title, date, or tag is factually correct.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path


root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the DemoLogic repository or notebooks/ folder")
sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root))

from notebooks.tools import show_table


from demo_logic.notebook_support import tagging_eligible_rows, title_word_count
from demo_logic.snapshot import load_snapshot
from demo_logic.taxonomy import topic_tags

In [ ]:
snapshot_rows, manifest = load_snapshot(root / "data/dataset.jsonl", root / "data/manifest.json")
TOPIC_TAGS = topic_tags(manifest["taxonomy_version"])
rows = tagging_eligible_rows(snapshot_rows)
excluded_title_rows = [row for row in snapshot_rows if title_word_count(row["title"]) < 2]
if not rows:
    raise ValueError(
        "No records have at least two title words. Inspect the snapshot before continuing."
    )

row_count = len(rows)
show_table(
    ["Dataset fact", "Value"],
    [
        ("Loaded taxonomy", manifest["taxonomy_version"]),
        ("Records in verified snapshot", len(snapshot_rows)),
        ("Excluded: fewer than two title words", len(excluded_title_rows)),
        ("Records eligible for tagging and EDA", row_count),
        ("Earliest event date", min(row["date"] for row in rows)),
        ("Latest event date", max(row["date"] for row in rows)),
    ],
)

## 2. Which records are included?

Preparation keeps active, dated demonstrations with one or two approved topic tags.
It excludes deleted records, records without a date, and records without a topic tag.
This notebook then excludes titles with fewer than two words from EDA and tagging.
`"Peace"` is excluded; `"Peace now"` is kept. Words are separated by whitespace;
punctuation alone does not count, and a hyphenated word counts as one. Notebook 02
checks that the saved split contains exactly the records allowed by this rule. Historical tags in the source stay unchanged.

These exclusions limit what we can conclude: this dataset does not represent every
untagged or incomplete record in production.

The table below reads the converter's recorded counts. Each excluded record is counted
under the first exclusion rule that applied. A missing source count means the manifest
did not record it; it does not mean zero.

In [ ]:
exclusions = manifest.get("excluded_counts", {})
exclusion_labels = {
    "deleted_demos": "Excluded: deleted records",
    "missing_date": "Excluded: records without a date",
    "untagged_demos": "Excluded: records without topic tags",
}
show_table(
    ["Preparation step", "Records"],
    [
        ("Records in source export", manifest.get("source_row_count", "Not recorded")),
        *[(exclusion_labels.get(reason, reason), count) for reason, count in exclusions.items()],
        ("Records in prepared snapshot", len(snapshot_rows)),
        ("Excluded here: fewer than two title words", len(excluded_title_rows)),
        ("Records used for EDA and tagging", row_count),
    ],
)

## 3. Which usable records still have missing information?

A record can pass validation and still lack useful information. For example, an empty
postcode is allowed, but a model using location would need a plan for that missing value.
The percentages below use **eligible records with at least two title words** as the denominator. A record can be
counted in more than one row.

Blank and one-word titles have already been excluded by the two-word rule. Missing dates, IDs, and
approved tags are handled earlier by preparation or validation; a zero here would not
mean that the production source had no missing values.

In [ ]:
missing_counts = {
    "Blank original title": sum(not row["title"].strip() for row in rows),
    "Missing postcode": sum(not (row["postcode"] or "").strip() for row in rows),
    "No venue tokens": sum(not any(token.strip() for token in row["venue_tokens"]) for row in rows),
    # The loader accepts strings or None. Whitespace is missing; midnight is valid.
    "Missing start time": sum(not (row["start_time"] or "").strip() for row in rows),
}
show_table(
    ["Information missing", "Records", "% of eligible records"],
    [(name, count, f"{100 * count / row_count:.1f}%") for name, count in missing_counts.items()],
)

## 4. Which titles and places need review?

Titles with fewer than two words have already been excluded, including one-character
placeholders. The exclusion count appears in sections 1 and 2. A title with two words
is eligible, but that alone does not prove its topic is clear or its historical tags are correct.

For places, we check the information available in this snapshot:
- **No venue information:** `venue_tokens` is empty or contains only blank strings.
  These tokens are the prepared words describing the venue; no tokens means the
  snapshot gives us no venue words to use.
- **No postcode:** the postcode is absent or blank. A venue may still be present.
- **Neither venue words nor postcode:** both are missing. Check the source to learn
  whether the place was never provided or was lost during preparation.

The rows below overlap; do not add them together. Section 7 lets you inspect examples.
Dates remain as provided in the source. A future date alone does not establish an error.
The final-test date policy is recorded separately in the saved split manifest.

In [ ]:
def has_no_venue(row):
    return not any(token.strip() for token in row["venue_tokens"])


def has_no_postcode(row):
    return not (row["postcode"] or "").strip()


quality_flags = {
    "No venue words in the snapshot": has_no_venue,
    "No postcode in the snapshot": has_no_postcode,
    "Neither venue words nor postcode": lambda row: has_no_venue(row) and has_no_postcode(row),
}
quality_counts = {name: sum(check(row) for row in rows) for name, check in quality_flags.items()}
show_table(
    ["Reason to review", "Records", "% of eligible records"],
    [(name, count, f"{100 * count / row_count:.1f}%") for name, count in quality_counts.items()],
)

## 5. Are some topic tags much rarer than others?

Here, **tag support** means the number of records carrying a tag. It is a count,
not a model score or proof that the tag is correct. Include all 12 approved tags,
even when a tag has zero records. A record with two tags contributes to two rows,
so the percentages can add up to more than 100%.

The bar chart uses one horizontal bar per tag. Longer bars mean more records;
the number beside each bar gives the exact count. The table includes percentages.

The chart uses Matplotlib from the locked notebook environment. If the import fails,
follow [notebook setup](../docs/notebook-setup.md) and select the DemoLogic kernel.

Small counts warn us that later per-tag results may be uncertain. This table alone
cannot tell us how many independent demonstrations those records represent.

In [ ]:
import matplotlib.pyplot as plt


tag_counts = Counter(tag for row in rows for tag in row["tag_slugs"])
show_table(
    ["Historical topic tag", "Records carrying tag", "% of eligible records"],
    [
        (tag, tag_counts[tag], f"{100 * tag_counts[tag] / row_count:.1f}%")
        for tag in sorted(TOPIC_TAGS, key=lambda tag: (tag_counts[tag], tag))
    ],
)

ordered_tags = sorted(TOPIC_TAGS, key=lambda tag: (tag_counts[tag], tag))
fig, ax = plt.subplots(figsize=(10, 6), layout="constrained")
bars = ax.barh(ordered_tags, [tag_counts[tag] for tag in ordered_tags])
ax.bar_label(bars, padding=4)
ax.set_title("How many records carry each historical topic tag?")
ax.set_xlabel("Number of demonstration records (a record can carry two tags)")
ax.set_ylabel("Historical topic tag")
ax.set_xlim(0, max(max(tag_counts.values(), default=0) * 1.15, 1))
ax.xaxis.get_major_locator().set_params(integer=True)
plt.show()
plt.close(fig)  # Release the figure after showing it, including during automated runs.

## 6. Are records duplicated, or are events part of a recurring series?

**Check A — Two records differ only in their event ID.** We compare every field in
the prepared snapshot except `external_id`. The title, normalized title, date, start
time, weekday, time bucket, postcode, venue tokens, existing tags, recurrence key,
and schema/taxonomy versions must all match. Even list order must match.

For example, imagine record `A` and record `B` both describe "Bicycle demonstration"
on 12 September at 14:00. If **all their other saved fields also match**, this check
flags them as a possible duplicate. It does not prove that one should be deleted.
Two equal titles alone are not enough. If their dates or tags differ, this exact
comparison does not flag them, even if a person might still consider them duplicates.

**Check B — Several records belong to the same recurring series.** The exporter
assigns a `recurrence_key` to link editions of a series. Four weekly bicycle
demonstrations can share that key while having four different dates. That is one
series with four records, and can be perfectly valid.

The table reports both checks separately. Review possible duplicates against the
source; do not remove recurring events simply because they share a title or series.
Notebook 02 inspects the saved event groups and checks that a group stays within one split.

In [ ]:
# load_snapshot reads validated JSONL: dates are strings, with no Python sets or custom objects.
# Sorted JSON keys make the comparison independent of dictionary order; list order still counts.
exact_content = Counter(
    json.dumps(
        {key: value for key, value in row.items() if key != "external_id"},
        sort_keys=True,
        ensure_ascii=False,
    )
    for row in rows
)
recurrence_counts = Counter(row["recurrence_key"] for row in rows)
repeat_summary = {
    "Possible duplicate sets: every field except ID matches": sum(
        count > 1 for count in exact_content.values()
    ),
    "Records in those possible duplicate sets": sum(
        count for count in exact_content.values() if count > 1
    ),
    "Exporter series with more than one record": sum(
        count > 1 for count in recurrence_counts.values()
    ),
    "Records in the largest exporter series": max(recurrence_counts.values(), default=0),
}
show_table(["Question", "Count"], repeat_summary.items())

## 7. Optional: inspect private examples of each problem

Set `SHOW_PRIVATE_EXAMPLES` to `True` for local inspection, then rerun this cell.
It shows up to three examples **per issue** from section 4 among the eligible records.
A record can appear under several issues. No examples for an issue means its count is zero.

Use the event ID to find the original record in your source export or application.
Check whether place information exists there. Excluded short titles do not appear in this sample.
If you need to inspect more records, increase `EXAMPLES_PER_ISSUE`. These first records
are a convenience sample; they do not establish how often a flag is a real error.
Clear the output before committing, since event titles and IDs are private here.

In [ ]:
SHOW_PRIVATE_EXAMPLES = False
EXAMPLES_PER_ISSUE = 3

if SHOW_PRIVATE_EXAMPLES:
    review_checks = {
        **quality_flags,
    }
    for issue, check in review_checks.items():
        examples = [row for row in rows if check(row)]
        show_table(
            ["Review issue", "Total flagged", "Shown"],
            [
                (issue, len(examples), min(len(examples), EXAMPLES_PER_ISSUE)),
            ],
        )
        show_table(
            ["Event ID", "Original title", "Date", "Postcode", "Venue tokens"],
            [
                (
                    row["external_id"],
                    row["title"],
                    row["date"],
                    row["postcode"],
                    row["venue_tokens"],
                )
                for row in examples[:EXAMPLES_PER_ISSUE]
            ],
        )
else:
    print(
        "Private examples are hidden. Set SHOW_PRIVATE_EXAMPLES = True to inspect flagged records."
    )

## 8. Save the measurements; record your decision separately

The next cell saves the counts, input fingerprint, and inclusion rules in a private
report. An identical rerun is allowed. If a report already exists with different
contents, keep it and choose a new report filename so earlier work is preserved.

The automated report is evidence for your decision. It does not declare the data
ready or resolve placeholder titles, missing places, possible duplicates, or labels.
This version records the two-word title eligibility rule and uses a `snapshot-quality-v3-...json`
filename to preserve reports saved by the earlier notebook.

In [ ]:
from demo_logic.notebook_support import save_private_json

quality_report = {
    "report_version": 3,
    "dataset_sha256": manifest["content_sha256"],
    "source_exported_at": manifest.get("source_exported_at"),
    "source_row_count": manifest.get("source_row_count"),
    "prepared_row_count": len(snapshot_rows),
    "eligible_row_count": row_count,
    "title_eligibility": {
        "minimum_words": 2,
        "word_count_rule": "whitespace-separated tokens containing a letter or digit",
        "excluded_count": len(excluded_title_rows),
    },
    "eligible_date_range": [min(row["date"] for row in rows), max(row["date"] for row in rows)],
    "inclusion_rules": [
        "active",
        "dated",
        "one or two approved topic tags",
        "at least two title words",
    ],
    "excluded_counts": exclusions,
    "missing_fields": missing_counts,
    "title_and_place_review": quality_counts,
    "records_by_year": dict(sorted(Counter(row["date"][:4] for row in rows).items())),
    "records_by_historical_tag": {tag: tag_counts[tag] for tag in sorted(TOPIC_TAGS)},
    "repeat_checks": repeat_summary,
    "decision": "pending_human_audit",
}
report_path = root / "reports" / f"snapshot-quality-v3-{manifest['content_sha256'][:12]}.json"
save_private_json(report_path, quality_report)
print(f"Private quality report saved: {report_path.relative_to(root)}")

## 9. Your next actions before Notebook 02

Running the cells measures the data. **Your task now is to decide how the known
problems should be handled**, and write down that decision. The two-word title rule is applied automatically. Other corrections or exclusions
require a separate decision.

1. **Record the title exclusion.** Copy the number excluded in section 2 into your
   audit. The agreed rule is already applied: only titles with at least two words
   enter EDA and the split. You do not need to remove these rows manually.
2. **Inspect missing places.** In section 7, set `SHOW_PRIVATE_EXAMPLES = True` and
   rerun the cell. Look up examples by event ID in the source. Is the place absent
   there, or did preparation lose it? Missing places alone need not exclude a record
   from a title-only model. Record your findings and any preparation fix needed.
3. **Check possible duplicates only if section 6 reports any.** Compare the saved
   fields and source records before deciding to exclude anything. Recurring editions
   on different dates can stay as separate records; Notebook 02 inspects their saved groups.
4. **Record rare tags as a limitation.** Use section 5 to note tags with little or no
   data. Do not delete a tag simply because it is rare. Notebook 02 will show whether
   each split has enough examples; no model has been evaluated at this point.
5. **Write your decision in `reports/YYYY-MM-DD.md`**, using today's Berlin date.
   Create the file if needed, or append a section if it already exists. Copy the full
   `dataset_sha256` from the private JSON report saved in section 8. Include the policy,
   evidence you inspected, any unresolved issue, and whether you can continue.

For example, adapt this private audit entry to what you actually found:

```markdown
## Snapshot quality review
- Quality report: reports/snapshot-quality-v3-<hash prefix>.json
- Full dataset_sha256: <copy from that report>
- Titles: <count> records excluded because their titles have fewer than two words.
  Policy: at least two title words are required for EDA and tagging.
- Places: <count> without venue words; <count> without both venue words and postcode.
  Finding: <missing at source or lost during preparation>.
  Policy: <for example, keep for the title-only model>.
- Possible duplicates: <none, or findings and decision>.
- Rare tags: <tag names and counts to watch in Notebook 02>.
- Decision: <continue with these limitations OR stop until named issues are resolved>.
- Follow-up: <specific correction needed, or none>.
```

**When are you done?** You have a saved quality report and a written decision covering
these issues. If you accept the snapshot as it is, restore its
[matching private benchmark bundle](../docs/data-preparation.md#restore-the-private-benchmark)
before continuing to [Notebook 02](02_label_audit_eda_and_splits.ipynb). If your decision requires additional corrections
or exclusions beyond the title rule, implement those in the preparation process, generate a new dataset and
matching manifest, and rerun this notebook before making the split. Writing a policy
in the audit does not apply it to the data. Do not edit `data/dataset.jsonl` by hand.

Finally, hide private examples again and clear all notebook outputs before committing.